In [1]:
import json

# Load sample_candidates.json (only 50 candidates, small file)
with open("../sample_candidates.json") as f:
    candidates = json.load(f)

print(f"Total candidates: {len(candidates)}")

# Question 1: How many are open to work?
open_to_work = [c for c in candidates if c["redrob_signals"]["open_to_work_flag"]]
print(f"Open to work: {len(open_to_work)}")

# Question 2: Average recruiter response rate
avg_response = sum(c["redrob_signals"]["recruiter_response_rate"] for c in candidates) / len(candidates)
print(f"Avg recruiter response rate: {avg_response:.2f}")

# Question 3: How many have NO GitHub linked?
no_github = [c for c in candidates if c["redrob_signals"]["github_activity_score"] == -1]
print(f"No GitHub: {len(no_github)}")

# Question 4: Most common skills
from collections import Counter
all_skills = []
for c in candidates:
    for s in c["skills"]:
        all_skills.append(s["name"])
print("\nTop 15 skills:")
for skill, count in Counter(all_skills).most_common(15):
    print(f"  {skill}: {count}")

# Question 5: Skill claim vs assessment score mismatch
print("\nSelf-reported vs assessed skill mismatches:")
for c in candidates:
    assessed = c["redrob_signals"]["skill_assessment_scores"]
    for skill in c["skills"]:
        if skill["name"] in assessed:
            if skill["proficiency"] in ["advanced", "expert"] and assessed[skill["name"]] < 50:
                print(f'  {c["candidate_id"]}: claims {skill["proficiency"]} {skill["name"]} but scored {assessed[skill["name"]]:.1f}')

Total candidates: 50
Open to work: 16
Avg recruiter response rate: 0.44
No GitHub: 33

Top 15 skills:
  AWS: 10
  Data Pipelines: 10
  Node.js: 9
  Sales: 9
  Figma: 9
  Tailwind: 8
  GCP: 8
  Content Writing: 8
  GraphQL: 8
  gRPC: 8
  Terraform: 8
  Scrum: 8
  Java: 8
  Spring Boot: 8
  Hadoop: 8

Self-reported vs assessed skill mismatches:
  CAND_0000001: claims advanced NLP but scored 38.8
  CAND_0000001: claims advanced Fine-tuning LLMs but scored 41.6
  CAND_0000011: claims advanced Recommendation Systems but scored 29.8
  CAND_0000025: claims advanced LangChain but scored 40.0
  CAND_0000027: claims advanced Data Science but scored 35.1


In [2]:
# Check honeypot-style contradictions
print("Potential honeypot signals:")
for c in candidates:
    sig = c["redrob_signals"]
    cid = c["candidate_id"]
    flags = []
    
    # Contradiction 1: open to work but very inactive
    from datetime import datetime, date
    last_active = datetime.strptime(sig["last_active_date"], "%Y-%m-%d").date()
    days_inactive = (date.today() - last_active).days
    if sig["open_to_work_flag"] and days_inactive > 180:
        flags.append(f"open_to_work but inactive {days_inactive} days")
    
    # Contradiction 2: high endorsements but zero connections
    if sig["endorsements_received"] > 50 and sig["connection_count"] < 10:
        flags.append(f"endorsements={sig['endorsements_received']} but connections={sig['connection_count']}")
    
    # Contradiction 3: perfect offer acceptance but zero interview completion
    if sig["offer_acceptance_rate"] == 1.0 and sig["interview_completion_rate"] < 0.1:
        flags.append("offer_acceptance=1.0 but never completes interviews")
    
    # Contradiction 4: 100% profile complete but unverified
    if sig["profile_completeness_score"] == 100 and not sig["verified_email"]:
        flags.append("completeness=100 but email unverified")

    if flags:
        print(f"\n  {cid}:")
        for f in flags:
            print(f"    ⚠ {f}")

Potential honeypot signals:

  CAND_0000002:
    ⚠ open_to_work but inactive 211 days

  CAND_0000005:
    ⚠ open_to_work but inactive 253 days

  CAND_0000036:
    ⚠ open_to_work but inactive 181 days


In [3]:
def is_hard_disqualified(candidate):
    companies = [r["company"] for r in candidate["career_history"]]
    consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]
    
    # All consulting, no product company
    all_consulting = all(any(c in comp for c in consulting) for comp in companies)
    if all_consulting:
        return True, "pure consulting background"
    
    # Inactive too long
    from datetime import datetime, date
    last_active = datetime.strptime(candidate["redrob_signals"]["last_active_date"], "%Y-%m-%d").date()
    if (date.today() - last_active).days > 180:
        return True, "inactive 6+ months"
    
    return False, None

In [4]:
disqualified = []
passed = []

for c in candidates:
    failed, reason = is_hard_disqualified(c)
    if failed:
        disqualified.append((c["candidate_id"], reason))
    else:
        passed.append(c)

print(f"Total: {len(candidates)}")
print(f"Disqualified: {len(disqualified)}")
print(f"Passed: {len(passed)}")
print("\nDisqualified list:")
for cid, reason in disqualified:
    print(f"  {cid}: {reason}")

Total: 50
Disqualified: 20
Passed: 30

Disqualified list:
  CAND_0000002: inactive 6+ months
  CAND_0000003: pure consulting background
  CAND_0000005: inactive 6+ months
  CAND_0000008: pure consulting background
  CAND_0000012: inactive 6+ months
  CAND_0000017: inactive 6+ months
  CAND_0000020: inactive 6+ months
  CAND_0000021: inactive 6+ months
  CAND_0000024: pure consulting background
  CAND_0000026: inactive 6+ months
  CAND_0000027: pure consulting background
  CAND_0000028: pure consulting background
  CAND_0000029: inactive 6+ months
  CAND_0000030: inactive 6+ months
  CAND_0000036: inactive 6+ months
  CAND_0000037: inactive 6+ months
  CAND_0000042: inactive 6+ months
  CAND_0000044: inactive 6+ months
  CAND_0000047: pure consulting background
  CAND_0000050: inactive 6+ months


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
test = model.encode(["test sentence"])
print("Model loaded, shape:", test.shape)

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded, shape: (1, 384)
